In [1]:
%run ../../Latex_macros.ipynb

<IPython.core.display.Latex object>

# RLLM-6: GRPO and its refinements

## Where this session fits

> **The limitation we start from: C2's plain GRPO has known weak spots, and published recipes show that the details decide whether it works.**

- **The question:** which weak spots of GRPO do later recipes fix, and how?
- **Position: session 6 of the RL for LLMs series (RLLM-1 to RLLM-13)**
    - RLLM-5 gave PPO in depth, including what the critic costs
    - this session is the other branch: GRPO without a critic, and the fixes proposed for it
    - the next session, RLLM-7, begins the "alternatives to GRPO" sessions
- **Series goal:** turn a pretrained language model into a useful, aligned assistant with reinforcement learning, and know the methods well enough to choose among them

**By the end you can**

1. point to each weak spot in the GRPO objective of C2
2. say what DAPO, Dr. GRPO and GSPO each change, and why
3. compute the weight a token gets under two loss-averaging rules
4. say what each fix risks, and what evidence the papers give
5. explain why a refinement can matter more than the algorithm's headline idea

**Papers used** (read for this session): the DAPO, Dr. GRPO and GSPO papers, cited on the last slide.


# The starting point: the GRPO objective and its weak spots

**Why this section?** To change an objective we need to see its parts.

> **The GRPO objective of C2 averages a clipped, token-level term over tokens and then over the group, with the same advantage for every token of a completion.**

$$
\frac{1}{G} \sum_{i=1}^{G} \frac{1}{\lvert \y_i \rvert} \sum_{\tt} \Bigl[ \min{} \bigl( r_{i,\tt}(\theta)\, \hat{\advseq}_i, \;\; \text{clip}\bigl(r_{i,\tt}(\theta),\, 1 - \epsilon_{\text{clip}},\, 1 + \epsilon_{\text{clip}}\bigr)\, \hat{\advseq}_i \bigr) \; - \; \beta \, \KL_{i,\tt} \Bigr]
\qquad \hat{\advseq}_i = \frac{\rew_i - \operatorname{mean}}{\operatorname{std}}
$$

- read it as parts
    - the **advantage** $\hat{\advseq}_i$: centered and divided by the group's standard deviation
    - the **averaging**: over the tokens of each completion, then over the group
    - the **clip**: the same range above and below $1$
    - the **KL term**
    - the **ratio** $r_{i,\tt}$: per token, though the reward is per completion

**Each part has a published refinement**

| Part | The weak spot | Change | Source |
|:--|:--|:--|:--|
| averaging | long completions' tokens count less | average over all tokens of the group, or over a constant | DAPO, Dr. GRPO |
| advantage scale | easy and hard questions are up-weighted | drop the division by the standard deviation | Dr. GRPO |
| clip | a tight upper clip limits rare tokens | a higher upper clip: $\epsilon_{\text{low}} \ne \epsilon_{\text{high}}$ | DAPO |
| sampling | all-equal groups give no gradient | filter them and sample more | DAPO |
| truncation | a cut-off completion may be unfairly scored | mask it, or penalize length gently | DAPO |
| KL term | a long-reasoning model must move far | drop it | DAPO, Dr. GRPO |
| ratio | a token ratio, a completion reward | a ratio for the whole completion | GSPO |

> **These changes are not independent, and each paper tested its own setting. Read them as options, not as a ranking.**


## Why details matter: the DAPO report

**Why this section?** The motivating fact: a naive GRPO run fell well short of a result DeepSeek reported.

> **The DAPO paper reports that a naive GRPO run on a 32-billion-parameter base model reached 30 points on AIME 2024, against 47 reported for DeepSeek-R1-Zero-Qwen-32B.**

- **DAPO** (Decoupled Clip and Dynamic sAmpling Policy Optimization): an open recipe and system from ByteDance and collaborators
- its paper reports 50 points on the same benchmark, using about half the training steps of the 47-point run
- the gap is closed by a short list of changes, found by watching training and diagnosing failures
    - the next slides take them one by one, with the paper's reasons
- how to read the numbers
    - one base model (Qwen2.5-32B), one benchmark (AIME 2024, averaged over 32 samples), the authors' own runs
    - they show the changes matter in that setting
    - they do not rank the changes in general


## Refinement 1: how the loss is averaged

**Why this section?** The averaging decides how much each token counts.

> **GRPO averages over tokens within a completion first, so a token in a long completion counts less than a token in a short one.**

- take $G = 2$ completions, of $10$ and $100$ tokens: the weight on each token in the objective is
| | completion A (10 tokens) | completion B (100 tokens) | ratio A : B |
|:--|:--|:--|:--|
| GRPO: $1/(G\,\lvert \y_i \rvert)$ | 0.0500 | 0.0050 | 10 : 1 |
| token-level (DAPO): $1/\sum_i \lvert \y_i \rvert$ | 0.0091 | 0.0091 | 1 : 1 |

- **token-level averaging** (DAPO): divide by the total number of tokens in the group, so every token counts the same
- **Dr. GRPO**: divide by a constant instead, so no per-completion length enters

**Why the paper says it matters**

- DAPO: with the per-completion average, tokens in long completions contribute less
    - good long reasoning is learned more slowly
    - long gibberish or repetition is penalized less, which the paper links to growing entropy and length
- Dr. GRPO: the same length factor biases the sign too
    - for a correct completion (advantage $> 0$), shorter ones are pushed up more
    - for an incorrect one (advantage $< 0$), longer ones are penalized less
    - so the policy drifts toward long wrong answers
- the DAPO ablation: this change added the least accuracy, but made training more stable (paper's words)


## Refinement 2: the advantage scale

**Why this section?** Dividing by the group's standard deviation re-weights questions.

> **Dr. GRPO argues that dividing by the group standard deviation gives questions that are almost all right or all wrong a larger weight.**

- with $G = 8$ and rewards in $\{0, 1\}$, the sample standard deviation and the factor $1/\text{std}$ are
  
| group | std | factor $1/\text{std}$ |
|:--|:--|:--|
| 1 of 8 correct | 0.35 | 2.83 |
| 4 of 8 correct | 0.53 | 1.87 |

- the question with one success in eight counts about $1.5$ times as much as the evenly split question
- Dr. GRPO calls this a **difficulty bias** and removes the division
    - the advantage is then just the reward minus the group mean
- DAPO and GSPO keep the standard deviation, so whether to drop it is an open choice
- common PPO practice normalizes over the whole batch; this normalization is per question


## Refinement 3: clip higher

**Why this section?** The clip of B4 is symmetric. The DAPO paper argues that is the wrong shape for exploration.

> **A ratio clip of $1 + \epsilon$ lets a likely token grow a lot, and lets a rare token grow only a little.**

- the paper's example, with $\epsilon = 0.2$ and a positive advantage: a token's new probability is at most $1.2 \times$ its old one
    - old probability $0.01$: at most $0.012$
    - old probability $0.9$: at most $1.080$ (above $1$: the clip does not bind)
- so a rare, exploratory token can barely rise in one step
- the paper observed **entropy collapse** with the standard clip: the sampled completions of a prompt become nearly the same

**The fix**

- decouple the two ends: clip to $[\,1 - \epsilon_{\text{low}},\; 1 + \epsilon_{\text{high}}\,]$
- the paper reports $\epsilon_{\text{low}} = 0.2$ and $\epsilon_{\text{high}} = 0.28$
    - it keeps $\epsilon_{\text{low}}$ because raising it would push low-probability tokens toward zero
- the paper reports a higher entropy and more varied samples with the change
- **a risk:** a wider range allows bigger steps, and the setting was tuned in one setting


## Refinement 4: do not waste groups, and do not punish cut-offs

**Why this section?** Two sampling-side fixes: what to do with groups that teach nothing, and with completions that were cut off.

> **If every completion in a group earns the same reward, the group's advantages are all zero and it contributes no gradient. DAPO filters such groups out and samples more.**

- **dynamic sampling:** keep a prompt only if its group has at least one success and at least one failure
    - sampling continues until the batch is full of informative groups
    - the paper notes more generation per step, but fewer steps to reach the same accuracy
- as training goes on, more prompts are answered correctly every time, so more groups are wasted

**Overlong completions**

- a completion cut off at the length limit may get a punishing reward, even if its reasoning was sound
    - the paper says this adds noise
- **overlong filtering:** mask the loss of cut-off completions
- **soft overlong punishment:** a penalty that grows linearly inside a final stretch before the limit
    - the paper's setting: a limit of $16\,384$ tokens and a $4\,096$-token stretch
    - it is added to the correctness reward


## Refinement 5: drop the KL term

**Why this section?** In C1 and RLLM-5 the KL kept the policy near a trusted model. Here two papers remove it.

> **The DAPO paper argues that when training a long reasoning model, the policy should move far from its start, so the KL restriction is unnecessary.**

- the two papers argue differently
    - DAPO: the reasoning model should move far from its start
    - Dr. GRPO, from verifiable rewards: a verifier removes the worry that the policy leaves the region where a reward model is accurate
        - it assumes $\beta = 0$ throughout, and notes that the reference model's memory and compute are saved
- in RLHF with a learned reward model the KL keeps the policy where that model is accurate (RLLM-2)
- without it, nothing anchors the policy except the reward and the clip
    - the benefit: no reference model in memory, and no extra log-probabilities to compute
    - the risk: drift the reward does not forbid
- **a design choice that depends on the reward, not a rule about GRPO**


## Refinement 6: a ratio for the whole completion

**Why this section?** The reward is one number per completion, but the ratio in the objective is per token.

> **GSPO defines the importance ratio, and the clip, on the whole completion rather than on each token.**

- the GSPO paper's argument: token-level ratios make the gradient noisy and unstable
    - the reward is per completion, so a per-completion ratio matches the unit of reward
- the sequence ratio is the **geometric mean of the token ratios**: $s_i(\theta) = \exp\bigl(\frac{1}{\lvert \y_i \rvert} \sum_{\tt} \log r_{i,\tt}(\theta)\bigr)$
    - length normalization keeps the ratio in one numerical range for short and long completions
- clipping then removes whole completions that moved too far
    - the paper's clip ranges are $3 \times 10^{-4}$ and $4 \times 10^{-4}$, far narrower than for a token ratio
- the advantage is the group advantage of C2

**What the paper reports, and what to hold back**

- the paper reports more stable training, and notes a benefit for mixture-of-experts models
    - it describes a workaround, routing replay, needed to train an MoE model with the token ratio, which GSPO makes unnecessary
- the paper omits the KL term from its analysis for brevity
- the paper's evidence is from its own models and benchmarks (Qwen)
- GSPO also has a token-level variant, for when per-token advantages are wanted


## Putting the recipes side by side

**Why this section?** One table so the changes can be seen against the original.

> **Each recipe changes a different subset of the parts. Reported here as the papers state them.**

| Part | GRPO (C2) | DAPO | Dr. GRPO | GSPO |
|:--|:--|:--|:--|:--|
| averaging | per completion, then group | all tokens in the group | a constant | per completion (sequence ratio) |
| advantage scale | divide by std | divide by std | no division | divide by std |
| clip | symmetric | $\epsilon_{\text{low}} = 0.2$, $\epsilon_{\text{high}} = 0.28$ | not the subject | on the sequence ratio, much narrower |
| equal-reward groups | kept | filtered out | not the subject | not the subject |
| cut-off completions | scored as usual | masked or softly penalized | not the subject | not the subject |
| KL term | in the loss | removed | removed ($\beta = 0$) | omitted from analysis |
| ratio level | token | token | token | sequence |

**What the DAPO ablation shows, and does not**

| change, added in order | AIME 2024 (the paper's figure) |
|:--|:--|
| naive GRPO | 30 |
| + overlong filtering | 36 |
| + clip higher | 38 |
| + soft overlong punishment | 41 |
| + token-level loss | 42 |
| + dynamic sampling (DAPO) | 50 |

- cumulative, in the authors' order: each step's gain depends on the steps before it
- one base model, one benchmark, one set of settings
- the table supports "these changes helped here", not "this is the best recipe"


## What the refinements leave open

**Why this section?** They fix GRPO's loss and sampling. They do not change what the signal is.

> **All four recipes keep one number per completion, from the group of the same prompt. The credit problem and the sample problem remain.**

- **credit:** every token of a completion still gets the same advantage (RLLM-5's contrast with PPO)
- **samples:** each batch comes from the current policy
    - generating it is paid for again at every step
- **signal:** one scalar reward per completion
    - richer signals (a teacher's token distribution, a critique) are not used
- RLLM-7 to RLLM-13 ask three questions of any such recipe
    - whose samples does it learn from?
    - what does the update consume?
    - how fine is the signal?
- the recipes here are the baseline those sessions compare against


# What we established, and what it sets up

- **Established**
    - the GRPO objective has parts: advantage scale, averaging, clip, sampling, KL term, ratio level
    - **DAPO:** token-level averaging, a higher upper clip, dynamic sampling, overlong handling, no KL
    - **Dr. GRPO:** no division by the length or by the standard deviation
    - **GSPO:** a ratio and clip on the whole completion
    - the evidence is from each paper's own setting: options, not a ranking
    - no refinement gives per-token credit or new kinds of samples
- **Sets up**
    - **the limitation:** one scalar per completion, from the current policy's own samples
    - RLLM-7 onward: alternatives that change whose samples are used, what the update consumes, and the grain of the signal
- **Papers**
    - DAPO: arXiv 2503.14476
    - Understanding R1-Zero-Like Training (Dr. GRPO): arXiv 2503.20783
    - GSPO: arXiv 2507.18071
